# Compute Fashion-CLIP Embeddings
Recompute image and text embeddings using `patrickjohncyh/fashion-clip` — a CLIP ViT-B/32 model fine-tuned on 800k fashion product images with rich attribute labels.

**Why Fashion-CLIP over standard CLIP?**
- Standard CLIP: trained on 400M general image-text pairs
- Fashion-CLIP: additionally fine-tuned on fashion-specific data → tighter clusters for same-product items

**Output files:**
- `fashion_clip_image_embeddings.pt` — `{str(itemId): tensor(512)}` for all train + phase1 items
- `fashion_clip_text_embeddings.pt` — `{str(itemId): tensor(512)}` for all train + phase1 items

After running this notebook, swap the paths in `train_and_predict.ipynb` and retrain.

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel

# ── Device ─────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_FOLDER    = "../data/"
IMAGES_FOLDER  = "../../images/"   # DO NOT LIST THIS FOLDER — open per-item only
IMAGE_OUT_PATH = "fashion_clip_image_embeddings.pt"
TEXT_OUT_PATH  = "fashion_clip_text_embeddings.pt"
MODEL_ID       = "patrickjohncyh/fashion-clip"

print(f"Will save image embeddings → {IMAGE_OUT_PATH}")
print(f"Will save text  embeddings → {TEXT_OUT_PATH}")


Device: mps
Will save image embeddings → fashion_clip_image_embeddings.pt
Will save text  embeddings → fashion_clip_text_embeddings.pt


In [2]:
# ── Load Fashion-CLIP model & processor ────────────────────────────────────────
print(f"Loading {MODEL_ID} ...")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model     = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

EMB_DIM = model.config.projection_dim   # 512 for ViT-B/32
print(f"  Embedding dim : {EMB_DIM}")
print(f"  Vision model  : {model.config.vision_config.model_type}")
print("Model loaded!")


Loading patrickjohncyh/fashion-clip ...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dim : 512
  Vision model  : clip_vision_model
Model loaded!


In [3]:
# ── Load item metadata for train + phase1 ─────────────────────────────────────
df_train  = pd.read_csv(os.path.join(DATA_FOLDER, "items_train.csv"))
df_phase1 = pd.read_csv(os.path.join(DATA_FOLDER, "items_phase_1.csv"))

# Combine — deduplicate by itemId so we encode each item once
df_all = pd.concat([df_train, df_phase1], ignore_index=True)
df_all = df_all.drop_duplicates(subset="itemId").reset_index(drop=True)
df_all["itemId_str"] = df_all["itemId"].astype(str)

df_all["title"]       = df_all["title"].fillna("")
df_all["description"] = df_all["description"].fillna("")
df_all["text"]        = (df_all["title"] + " " + df_all["description"]).str.strip()

print(f"Train items  : {len(df_train):,}")
print(f"Phase1 items : {len(df_phase1):,}")
print(f"Total unique : {len(df_all):,}")


Train items  : 928,234
Phase1 items : 199,835
Total unique : 1,128,069


In [ ]:
# ── Image embeddings ──────────────────────────────────────────────────────────
# Skip items whose images are already in the output file (resume-safe)
if os.path.exists(IMAGE_OUT_PATH):
    image_embeddings = torch.load(IMAGE_OUT_PATH, map_location="cpu", weights_only=False)
    print(f"Resuming — already have {len(image_embeddings):,} image embeddings")
else:
    image_embeddings = {}

BATCH_SIZE = 64
item_ids  = df_all["itemId_str"].tolist()
remaining = [iid for iid in item_ids if iid not in image_embeddings]
print(f"Items to encode: {len(remaining):,}")

def load_image_safe(item_id):
    path = os.path.join(IMAGES_FOLDER, f"{item_id}.jpg")
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

for batch_start in tqdm(range(0, len(remaining), BATCH_SIZE), desc="Image embeddings"):
    batch_ids  = remaining[batch_start : batch_start + BATCH_SIZE]
    batch_imgs = []
    valid_ids  = []

    for iid in batch_ids:
        img = load_image_safe(iid)
        if img is not None:
            batch_imgs.append(img)
            valid_ids.append(iid)

    if not batch_imgs:
        for iid in batch_ids:
            image_embeddings[iid] = torch.zeros(EMB_DIM)
        continue

    inputs = processor(images=batch_imgs, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        # Use vision_model directly → pooler_output → project → normalize
        vision_outputs = model.vision_model(**inputs)
        pooled = vision_outputs.pooler_output          # (B, hidden_dim)
        img_feats = model.visual_projection(pooled)    # (B, 512)
        img_feats = F.normalize(img_feats, dim=-1).cpu()

    for iid, feat in zip(valid_ids, img_feats):
        image_embeddings[iid] = feat

    # Fill zero vectors for any images that failed to load
    for iid in batch_ids:
        if iid not in image_embeddings:
            image_embeddings[iid] = torch.zeros(EMB_DIM)

    # Save checkpoint every 5k items
    if (batch_start // BATCH_SIZE) % 80 == 0:
        torch.save(image_embeddings, IMAGE_OUT_PATH)

torch.save(image_embeddings, IMAGE_OUT_PATH)
print(f"\nSaved {len(image_embeddings):,} image embeddings → {IMAGE_OUT_PATH}")


Items to encode: 1,128,069


Image embeddings:   0%|          | 0/17627 [00:00<?, ?it/s]

In [ ]:
# ── Text embeddings ───────────────────────────────────────────────────────────
if os.path.exists(TEXT_OUT_PATH):
    text_embeddings = torch.load(TEXT_OUT_PATH, map_location="cpu", weights_only=False)
    print(f"Resuming — already have {len(text_embeddings):,} text embeddings")
else:
    text_embeddings = {}

remaining_text = [iid for iid in item_ids if iid not in text_embeddings]
texts_map      = df_all.set_index("itemId_str")["text"].to_dict()
print(f"Items to encode: {len(remaining_text):,}")

for batch_start in tqdm(range(0, len(remaining_text), BATCH_SIZE), desc="Text embeddings"):
    batch_ids   = remaining_text[batch_start : batch_start + BATCH_SIZE]
    batch_texts = [texts_map.get(iid, "") for iid in batch_ids]

    inputs = processor(
        text=batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    ).to(device)

    with torch.no_grad():
        # Use text_model directly → pooler_output → project → normalize
        text_outputs = model.text_model(**inputs)
        pooled = text_outputs.pooler_output             # (B, hidden_dim)
        txt_feats = model.text_projection(pooled)       # (B, 512)
        txt_feats = F.normalize(txt_feats, dim=-1).cpu()

    for iid, feat in zip(batch_ids, txt_feats):
        text_embeddings[iid] = feat

    # Save checkpoint every 5k items
    if (batch_start // BATCH_SIZE) % 80 == 0:
        torch.save(text_embeddings, TEXT_OUT_PATH)

torch.save(text_embeddings, TEXT_OUT_PATH)
print(f"\nSaved {len(text_embeddings):,} text embeddings → {TEXT_OUT_PATH}")


In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────────
import random

print("=== Sanity check: cosine similarity between same-label pairs ===\n")

# Sample 5 random labels that appear ≥2 times in train
label_counts = df_train["label"].value_counts()
multi_labels = label_counts[label_counts >= 2].index.tolist()
sample_labels = random.sample(multi_labels, min(5, len(multi_labels)))

for lbl in sample_labels:
    ids = df_train[df_train["label"] == lbl]["itemId"].astype(str).tolist()
    id_a, id_b = ids[0], ids[1]

    emb_a = image_embeddings.get(id_a, torch.zeros(EMB_DIM))
    emb_b = image_embeddings.get(id_b, torch.zeros(EMB_DIM))
    img_sim = F.cosine_similarity(emb_a.unsqueeze(0), emb_b.unsqueeze(0)).item()

    txt_a = text_embeddings.get(id_a, torch.zeros(EMB_DIM))
    txt_b = text_embeddings.get(id_b, torch.zeros(EMB_DIM))
    txt_sim = F.cosine_similarity(txt_a.unsqueeze(0), txt_b.unsqueeze(0)).item()

    print(f"label={lbl}  items {id_a} & {id_b}")
    print(f"  image cosine sim : {img_sim:.4f}")
    print(f"  text  cosine sim : {txt_sim:.4f}\n")

print("If image/text sim is consistently high (>0.8) for same-label pairs → embeddings are working correctly.")


## How to use in `train_and_predict.ipynb`

Replace the two loading cells with:

```python
# Image embeddings
clip_embeddings_dict = torch.load("fashion_clip_image_embeddings.pt", map_location="cpu", weights_only=False)

# Text embeddings  
TEXT_EMB_DIM = 512   # same dim as before
text_embeddings_dict = torch.load("fashion_clip_text_embeddings.pt", map_location="cpu", weights_only=False)
```

The embedding dimension stays **512** (Fashion-CLIP is ViT-B/32 based), so **no model architecture changes needed** — just reinitialise the siamese model and retrain.